# Fine-tune Granite Multi-task Model

This notebook fine-tunes Granite on both security analysis and recruiter generation tasks.

## 1. Install Dependencies

In [ ]:
!pip install torch==2.1.0
!pip install transformers==4.36.0
!pip install datasets==2.16.0
!pip install peft==0.7.0
!pip install accelerate==0.25.0
!pip install bitsandbytes==0.41.3
!pip install trl==0.7.4

## 2. Setup

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 3. Configuration

In [ ]:
# Model config
MODEL_NAME = "ibm-granite/granite-3b-code-instruct"
OUTPUT_DIR = "../models/granite-multitask-finetuned"
DATASET_PATH = "../datasets/generated/combined_training_data.jsonl"

# Training config
LEARNING_RATE = 2e-4
NUM_EPOCHS = 3
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
MAX_SEQ_LENGTH = 2048

# LoRA config
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

print("✅ Configuration set")

## 4. Load Dataset

In [ ]:
dataset = load_dataset('json', data_files=DATASET_PATH, split='train')
print(f"Total examples: {len(dataset)}")
print("\nFirst example:")
print(dataset[0])

## 5. Format Dataset

In [ ]:
def format_for_training(example):
    messages = example['messages']
    text = ""
    for msg in messages:
        role = msg['role']
        content = msg['content']
        text += f"<|{role}|>\n{content}\n\n"
    return {"text": text}

dataset = dataset.map(format_for_training, remove_columns=dataset.column_names)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

train_dataset = dataset['train']
eval_dataset = dataset['test']

print(f"Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")

## 6. Load Model with Quantization

In [ ]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Configuring 4-bit quantization...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model)
print("✅ Model loaded")

## 7. Configure LoRA

In [ ]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 8. Setup Training

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    gradient_checkpointing=True,
    optim="paged_adamw_32bit",
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=10,
    save_strategy="epoch",
    evaluation_strategy="epoch",
    fp16=True,
    push_to_hub=False,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=lora_config,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    tokenizer=tokenizer,
    args=training_args,
    packing=False
)

print("✅ Trainer ready")

## 9. Start Training

In [ ]:
%%time
print("="*60)
print("STARTING FINE-TUNING")
print("="*60)

trainer.train()

print("\n" + "="*60)
print("TRAINING COMPLETE!")
print("="*60)

## 10. Save Model

In [ ]:
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Model saved to {OUTPUT_DIR}")

## 11. Upload to S3

In [ ]:
import os

# Set your S3 secret key
os.environ['S3_SECRET_KEY'] = 'your-secret-key-here'

!python ../scripts/upload_to_s3.py --model-path {OUTPUT_DIR} --verify

## Summary

✅ Fine-tuning complete!
✅ Model saved locally and uploaded to S3

Next: Deploy InferenceService in RHOAI